# Convolutional Neural Networks

Import of the used packages and methods

In [1]:
import numpy as np
import tensorflow as tf
from pyparsing import Dict
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import pandas as pd

I0000 00:00:1787148514.593092  698722 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787148514.593487  698722 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787148514.621466  698722 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787148515.282272  698722 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

# Split Train/Test in 5 folds
splitted to Dicts of:
- meta_tr (train)
- meta_te (test)

and each group will appear exactly once in the test set across all folds

In [2]:
from sklearn.model_selection import GroupKFold
from auswertung_features.build_features import split_into_label_blocks, load_session

meta = pd.read_csv("../auswertung_features/meta.csv")
gkf = GroupKFold(n_splits=5, shuffle=True, random_state=42)
meta_tr: dict[int, pd.DataFrame] = {}
meta_te: dict[int, pd.DataFrame] = {}

dict_blocks = {}
meta_blocks = {"subject_id":[], "hand": [], "block_id":[]}
for i, r in meta.iterrows():
        csv_path = r["rel_path"]
        df_session = load_session(csv_path)

        #Error Handling für leere dfs
        if df_session is None:
            continue

        blocks = split_into_label_blocks(df_session)
        for block in blocks:
            meta_blocks["subject_id"].append(r["subject_id"])
            meta_blocks["hand"].append(r["hand"])
            meta_blocks["block_id"].append(len(dict_blocks))
            dict_blocks[len(dict_blocks)] = block
meta_blocks = pd.DataFrame(meta_blocks)

for fold, (tr_idx, te_idx) in enumerate(gkf.split(meta_blocks, groups=meta_blocks["subject_id"])):
    meta_tr[fold] = meta_blocks.iloc[tr_idx].reset_index(drop=True)
    meta_te[fold] = meta_blocks.iloc[te_idx].reset_index(drop=True)


[WARN] ../ArduinoApps/emg_messung/python/messdaten/messung_l_170.csv: is empty


## Convert dataset to match TensorFlow Conventions

Input needs to be in shape (batch, steps, features). My own blocks (ndarray) are in the shape of (sample_length, sensor_channels). The steps are the sequence length which should be learned.

In [3]:
def build_xy_from_blocks(meta_df, dict_blocks):
    X_list, y_list = [], []
    for _, row in meta_df.iterrows():
        Xb, yb = dict_blocks[int(row["block_id"])]
        X_list.append(np.asarray(Xb, dtype=np.float32))
        y_list.append(int(yb))   # bleibt 0..4
    return X_list, np.asarray(y_list, dtype=np.int32)


In [4]:
def make_dataset(X_list, y, batch_size=32, training=True):
    ds = tf.data.Dataset.from_tensor_slices((X_list, y))
    if training:
        ds = ds.shuffle(buffer_size=len(X_list), reshuffle_each_iteration=True)

    ds = ds.padded_batch(
        batch_size=batch_size,
        padded_shapes=([None, 4], []),   # (steps, features), label scalar
        padding_values=(0.0, 0)
    ).prefetch(tf.data.AUTOTUNE)
    return ds


In [5]:
fold = 0
X_tr, y_tr = build_xy_from_blocks(meta_tr[fold], dict_blocks, labels_are_1_to_4=True)
X_te, y_te = build_xy_from_blocks(meta_te[fold], dict_blocks, labels_are_1_to_4=True)

ds_tr = make_dataset(X_tr, y_tr, batch_size=32, training=True)
ds_te = make_dataset(X_te, y_te, batch_size=32, training=False)

# Shape-Check
for xb, yb in ds_tr.take(1):
    print("xb:", xb.shape, "yb:", yb.shape)  # xb z.B. (32, 1190, 4)


TypeError: build_xy_from_blocks() got an unexpected keyword argument 'labels_are_1_to_4'

# Model class, 1d CNN

In [ ]:
from tensorflow.keras import layers, models
import tensorflow as tf

inputs = layers.Input(shape=(None, 4))
x = layers.Conv1D(32, 7, padding="same", activation="relu")(inputs)
x = layers.MaxPooling1D(2)(x)
x = layers.Conv1D(64, 5, padding="same", activation="relu")(x)
x = layers.GlobalAveragePooling1D()(x)
outputs = layers.Dense(4, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.fit(ds_tr, validation_data=ds_te, epochs=20)
